In [1]:
from data import load_data
import pandas as pd
from preprocessing import preprocess_data
from config import train_data_path, train_sql, models
from models import build_pipeline, model_config_builder
from hyperparameter_optimizer import randomize_search

In [2]:
df_train = load_data(train_data_path, train_sql, query = False)
X_train, y_train = preprocess_data(df_train)
start_distribution = {
    "logreg": {
        "logreg__class_weight": ["balanced"],
        "logreg__solver": ["saga"],
        "logreg__max_iter": [100, 500, 1000, 2000],
    },
    "rf": {
        "rf__n_estimators": [100, 200, 300, 500, 1000],
        "rf__max_depth": [None, 4, 6, 8, 10],
        "rf__min_samples_split": [2, 10, 20, 50],
        "rf__min_samples_leaf": [1, 5, 10, 20],
        "rf__max_features": ["sqrt", 0.3, 0.5],
        "rf__class_weight": ["balanced_subsample"],
    },
    "lightgbm": {
        "lightgbm__n_estimators": [50, 100, 200, 300, 400],
        "lightgbm__learning_rate": [0.01, 0.03, 0.05, 0.07],
        "lightgbm__num_leaves": [63, 100, 128, 180, 256],
        "lightgbm__max_depth": [-1, 6, 8, 10],
        "lightgbm__subsample": [0.5, 0.6, 0.7, 0.8],
        "lightgbm__colsample_bytree": [0.5, 0.6, 0.7, 0.8],
        "lightgbm__min_child_samples": [2, 5, 10, 20],
        "lightgbm__reg_lambda": [0.0, 0.5, 1],
        "lightgbm__reg_alpha": [0.0, 1, 5, 10],
    },
}

In [3]:
models_built = model_config_builder(models, set_params = False)
optimal_params : dict[str, dict[str, float]] = {}
for model in models_built:
    distribution = start_distribution[model]
    pipe = build_pipeline(model, models_built[model])
    best_params, best_score = randomize_search(pipe, distribution, X_train, y_train['readmit_30d'], random_state = 42)
    optimal_params[model] = best_params
    print(f"{model}: best score is {best_score}")

d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=50. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 4 candidates, totalling 20 fits


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


logreg: best score is 0.06277071396607452
Fitting 5 folds for each of 50 candidates, totalling 250 fits
rf: best score is 0.10948191840436641
Fitting 5 folds for each of 50 candidates, totalling 250 fits
[LightGBM] [Info] Number of positive: 1967, number of negative: 197565
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009532 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1447
[LightGBM] [Info] Number of data points in the train set: 199532, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.009858 -> initscore=-4.609558
[LightGBM] [Info] Start training from score -4.609558
lightgbm: best score is 0.13896196232121838


In [4]:
optimal_params

{'logreg': {'logreg__solver': 'saga',
  'logreg__max_iter': 100,
  'logreg__class_weight': 'balanced'},
 'rf': {'rf__n_estimators': 1000,
  'rf__min_samples_split': 10,
  'rf__min_samples_leaf': 5,
  'rf__max_features': 0.3,
  'rf__max_depth': None,
  'rf__class_weight': 'balanced_subsample'},
 'lightgbm': {'lightgbm__subsample': 0.7,
  'lightgbm__reg_lambda': 1,
  'lightgbm__reg_alpha': 0.0,
  'lightgbm__num_leaves': 63,
  'lightgbm__n_estimators': 300,
  'lightgbm__min_child_samples': 10,
  'lightgbm__max_depth': -1,
  'lightgbm__learning_rate': 0.01,
  'lightgbm__colsample_bytree': 0.7}}